# Logging

- Logging is a means of tracking "events" when your application runs. 

- An *event* can be anything of interest that happens during the execution of your program like occurance of an error, a simple infomatic message like your program started or you API call was successful etc.

  - Events are logged with a descriptive message which optionally can have associated application data.

  - Events also have an importance which you as the developer ascribes it; the importance can also be called the **level or severity**.

| Level     | When it’s used                                                                                  |
|-----------|-------------------------------------------------------------------------------------------------|
| `DEBUG`   | Detailed information, typically of interest only when diagnosing problems.                      |
| `INFO`    | Confirmation that things are working as expected.                                               |
| `WARNING` | An indication that something unexpected happened, or indicative of some problem in the near future (e.g. ‘disk space low’). The software is still working as expected. |
| `ERROR`   | Due to a more serious problem, the software has not been able to perform some function.         |
| `CRITICAL`| A serious error, indicating that the program itself may be unable to continue running.          |

<p align="right"><i>Source: <a href="https://docs.python.org/3/howto/logging.html">Python Logging Docs</a></i></p>



Setting up logging can be hard, but in this notebook we will go through different tools and libraries that can help you set up logging in your application.

## Python's `logging` module

In [7]:
# Setup logging in the simplest way possible

import os
import sys
import logging


# get log level from environment variable
LOG_LEVEL = os.environ.get("LOG_LEVEL", "INFO").upper()

# configure logger object
logging.basicConfig(
    format="{asctime} | {levelname} | {name}:{lineno}:{funcName} | {message}",
    style="{",  # uses {} as placeholders
    level=LOG_LEVEL,
    stream=sys.stdout,  # where to write the log messages, in this case stdout or console
)
logger = logging.getLogger(__name__)  # create logger object with the name of the current module

# log some messages
logger.debug("This is a debug message")  # this will not be printed because the log level is set to INFO
logger.info("This is an info message")
logger.warning("This is a warning message")
logger.error("This is an error message")
logger.critical("This is a critical message")
# logger.exception("This is an exception message")

2024-08-09 23:35:48,389 | INFO | __main__:22:<module> | This is an info message
2024-08-09 23:35:48,390 | WARNING | __main__:23:<module> | This is a warning message
2024-08-09 23:35:48,392 | ERROR | __main__:24:<module> | This is an error message
2024-08-09 23:35:48,392 | CRITICAL | __main__:25:<module> | This is a critical message


The logging library takes a modular approach and offers several categories of components: *loggers, handlers, filters, and formatters*.

- ***Loggers*** expose the interface that application code directly uses.
- ***Handlers*** send the log records (created by loggers) to the appropriate destination.
- ***Filters*** provide a finer grained facility for determining which log records to output.
- ***Formatters*** specify the layout of log records in the final output.

<p align="right"><i>Source: <a href="https://docs.python.org/3/howto/logging.html#advanced-logging-tutorial">Advanced Python Logging Docs</a></i></p>

In [1]:
# Advanced logging setup with a console handler and a formatter

import logging
import os
import sys


LOG_LEVEL = os.environ.get("LOG_LEVEL", "INFO").upper()

# create the logger
logger = logging.getLogger(__name__)
logger.setLevel(LOG_LEVEL)

# create a console handler and set its log level
ch = logging.StreamHandler(stream=sys.stderr)
ch.setLevel(LOG_LEVEL)

# create a formatter
formatter = logging.Formatter("{asctime} | {levelname} | {name}:{lineno}:{funcName} | {message}", style="{")
ch.setFormatter(formatter)  # add the formatter to the console handler

# add the console handler to the logger
logger.addHandler(ch)


# log some messages
logger.debug("This is a debug message")
logger.info("This is an info message")
logger.warning("This is a warning message")
logger.error("This is an error message")
logger.critical("This is a critical message")

2024-08-09 23:39:19,011 | INFO | __main__:28:<module> | This is an info message
2024-08-09 23:39:19,013 | WARNING | __main__:29:<module> | This is a warning message
2024-08-09 23:39:19,014 | ERROR | __main__:30:<module> | This is an error message
2024-08-09 23:39:19,014 | CRITICAL | __main__:31:<module> | This is a critical message


In [23]:
import random
import time

def process_user_actions(user_id, actions):
    """
    Simulates processing a list of user actions.
    
    Parameters:
    user_id (int): The ID of the user.
    actions (list of str): A list of actions to process.
    
    Returns:
    None
    """
    
    # Simulate different levels of logging for demonstration
    logger.info(f"Start processing actions for user {user_id}.")
    
    for index, action in enumerate(actions):
        try:
            logger.debug(f"Processing action {index + 1}/{len(actions)}: {action}.")
            
            # Simulate processing time
            processing_time = random.uniform(0.1, 0.5)
            time.sleep(processing_time)
            
            # Simulate a warning scenario
            if "warning" in action:
                logger.warning(f"Potential issue detected in action: {action}.")
            
            # Simulate an error scenario
            if "error" in action:
                raise ValueError(f"Failed to process action: {action}.")
            
            logger.info(f"Successfully processed action: {action}.")
        
        except Exception as e:
            logger.error(f"Error processing action {action}: {str(e)}")
            logger.exception(f"Exception details: {str(e)}")
            logger.debug("Continuing with the next action.")
    
    logger.info(f"Finished processing actions for user {user_id}.")

In [ ]:
# Sample usage
if __name__ == "__main__":
    user_id = 123
    actions = ["login", "view_page", "add_to_cart", "warning: slow network", "checkout", "error: payment failed"]
    
    process_user_actions(user_id, actions)

## Loguru

In [18]:
import sys
import json
from loguru import logger


def serialize_extra_keys(record):
    extra = record["extra"]
    if extra:
        record["extra"] = json.dumps(extra)
    return record

logging_config = {
    "handlers": [
        {
            "sink": sys.stdout,
            "format": "<green>{time:YYYY-MM-DD HH:mm:ss}</green> | <level>{level: <8}</level> | <cyan>{name}:{function}:{line}</cyan> | <level>{message}</level> | <level>{extra}</level>",
            "level": "INFO",
            "filter": serialize_extra_keys,
            "colorize": True,
        },
    ],
}

# Remove the default logger config and add custom configurations
logger.remove()
logger.configure(**logging_config)


# log some messages
logger.debug("This is a debug message")
logger.success("This is a success message")
logger.info("This is an info message")
logger.warning("This is a warning message")
logger.error("This is an error message")
logger.critical("This is a critical message")
logger.info(
    "Adding extra data to log messages",
    extra={"extra_key": "extra_value", "another_key": "another_value", "user_id": 12345}
)
# logger.exception("This is an exception message", extra={"user_id": 12345})


2024-08-09 23:59:44 | SUCCESS  | __main__:<module>:31 | This is a success message | {}
2024-08-09 23:59:44 | INFO     | __main__:<module>:32 | This is an info message | {}
2024-08-09 23:59:44 | WARNING  | __main__:<module>:33 | This is a warning message | {}
2024-08-09 23:59:44 | ERROR    | __main__:<module>:34 | This is an error message | {}
2024-08-09 23:59:44 | CRITICAL | __main__:<module>:35 | This is a critical message | {}
2024-08-09 23:59:44 | INFO     | __main__:<module>:36 | Adding extra data to log messages | {"extra": {"extra_key": "extra_value", "another_key": "another_value", "user_id": 12345}}


In [20]:
# Sample usage
if __name__ == "__main__":
    user_id = 123
    actions = ["login", "view_page", "add_to_cart", "warning: slow network", "checkout", "error: payment failed"]
    
    process_user_actions(user_id, actions)

2024-08-10 00:01:24 | INFO     | __main__:process_user_actions:17 | Start processing actions for user 123. | {}
2024-08-10 00:01:25 | INFO     | __main__:process_user_actions:35 | Successfully processed action: login. | {}
2024-08-10 00:01:25 | INFO     | __main__:process_user_actions:35 | Successfully processed action: view_page. | {}
2024-08-10 00:01:25 | INFO     | __main__:process_user_actions:35 | Successfully processed action: add_to_cart. | {}
2024-08-10 00:01:26 | WARNING  | __main__:process_user_actions:29 | Potential issue detected in action: warning: slow network. | {}
2024-08-10 00:01:26 | INFO     | __main__:process_user_actions:35 | Successfully processed action: warning: slow network. | {}
2024-08-10 00:01:26 | INFO     | __main__:process_user_actions:35 | Successfully processed action: checkout. | {}
2024-08-10 00:01:26 | ERROR    | __main__:process_user_actions:38 | Error processing action error: payment failed: Failed to process action: error: payment failed. | {}
202

## `aws-lambda-powertools`

In [7]:
from typing import Any
from aws_lambda_powertools import Logger
from aws_lambda_powertools.utilities.typing import LambdaContext
import os

LOG_LEVEL = os.getenv('LOG_LEVEL', 'INFO')

logger = Logger(level=LOG_LEVEL)

@logger.inject_lambda_context
def handler(event: dict, context: LambdaContext) -> Any:
    logger.info("This is an info message")
    logger.debug("This is a debug message")
    logger.append_keys(extra="additional data")
    
    return {"statusCode": 200, "body": "Hello, World!"}


In [ ]:
if __name__ == "__main__":
    handler({}, None)

In [25]:
from aws_lambda_powertools import Logger
from aws_lambda_powertools.utilities.typing import LambdaContext
from rich import print

# Initialize the logger
logger = Logger()


def handler(event: dict, context: LambdaContext):
    logger.info("Starting to process the event.")
    
    user_id = event.get("user_id", "unknown")
    actions = event.get("actions", [])
    
    logger.append_keys(user_id=user_id)  # Adding context to all logs
    # for action in actions:
    #     logger.info(f"Processing action: {action}")
        
    #     if "warning" in action:
    #         logger.warning(f"Potential issue with action: {action}")
        
    #     if "error" in action:
    #         logger.error(f"Error encountered in action: {action}")
    
    # logger.info("Finished processing the event.")
    process_user_actions(user_id, actions)
    return {"statusCode": 200, "body": "Handler executed successfully."}

In [26]:
# Simulate an event
if __name__ == "__main__":
    sample_event = {
        "user_id": 123,
        "actions": ["login", "view_page", "warning: slow network", "error: payment failed"]
    }
    
    # Simulate the Lambda context (optional)
    class Context:
        def __init__(self):
            self.function_name = "test-function"
            self.memory_limit_in_mb = 128
            self.invoked_function_arn = "arn:aws:lambda:us-west-2:123456789012:function:test-function"
            self.aws_request_id = "fake-request-id"
    
    context = Context()
    
    # Call the handler function as Lambda would
    response = handler(sample_event, context)
    print(response)

{"level":"INFO","location":"handler:10","message":"Starting to process the event.","timestamp":"2024-08-10 00:13:45,227+0530","service":"service_undefined","user_id":123}
{"level":"INFO","location":"process_user_actions:17","message":"Start processing actions for user 123.","timestamp":"2024-08-10 00:13:45,228+0530","service":"service_undefined","user_id":123}
{"level":"INFO","location":"process_user_actions:35","message":"Successfully processed action: login.","timestamp":"2024-08-10 00:13:45,651+0530","service":"service_undefined","user_id":123}
{"level":"INFO","location":"process_user_actions:35","message":"Successfully processed action: view_page.","timestamp":"2024-08-10 00:13:46,027+0530","service":"service_undefined","user_id":123}
{"level":"WARNING","location":"process_user_actions:29","message":"Potential issue detected in action: warning: slow network.","timestamp":"2024-08-10 00:13:46,177+0530","service":"service_undefined","user_id":123}
{"level":"INFO","location":"process_

{'statusCode': 200, 'body': 'Handler executed successfully.'}